In [1]:
# --- Pickle / Torch Checkpoint Inspector (Jupyter cell) ---
import os, sys, math
from collections import OrderedDict
from typing import Any
try:
    import numpy as np
except Exception:
    np = None
try:
    import torch
except Exception:
    torch = None
try:
    import jax.numpy as jnp
except Exception:
    jnp = None
try:
    import pandas as pd
except Exception:
    pd = None

def _is_namedtuple_instance(x): return isinstance(x, tuple) and hasattr(x, "_fields")
def _short(s: str, n: int) -> str: s = str(s); return s if len(s) <= n else s[: n - 1] + "…"
def _shape_dtype_device(x):
    if torch is not None and isinstance(x, torch.Tensor):
        dev  = f", device={x.device}" if (hasattr(x,'device') and str(x.device)!='cpu') else ""
        grad = ", requires_grad=True" if getattr(x, "requires_grad", False) else ""
        return f"shape={tuple(x.shape)}, dtype={x.dtype}{dev}{grad}"
    if np is not None and isinstance(x, np.ndarray): return f"shape={x.shape}, dtype={x.dtype}"
    if jnp is not None and isinstance(x, jnp.ndarray): return f"shape={x.shape}, dtype={x.dtype}"
    if pd is not None:
        if isinstance(x, pd.DataFrame): return f"DataFrame: shape={x.shape}, columns={list(x.columns)}"
        if isinstance(x, pd.Series):    return f"Series: shape={x.shape}, dtype={x.dtype}" + (f", name={x.name}" if x.name is not None else "")
    return ""
def _summ_stats(x):
    try:
        if torch is not None and isinstance(x, torch.Tensor):
            y = x.detach().float().cpu(); return f"min={float(y.min())}, max={float(y.max())}, mean={float(y.mean())}"
        if np is not None and isinstance(x, np.ndarray) and np.issubdtype(x.dtype, np.number):
            return f"min={float(np.min(x))}, max={float(np.max(x))}, mean={float(np.mean(x))}"
        if jnp is not None and isinstance(x, jnp.ndarray) and np is not None:
            y = np.array(x); 
            if np.issubdtype(y.dtype, np.number): return f"min={float(y.min())}, max={float(y.max())}, mean={float(y.mean())}"
    except Exception: pass
    return ""
def _describe_scalar(x, truncate):
    if isinstance(x, (int,float,complex,bool)): return repr(x)
    if isinstance(x, (str,bytes)):
        s = x.decode("utf-8", errors="replace") if isinstance(x, bytes) else x
        return repr(_short(s, truncate))
    return _short(repr(x), truncate)
def _iter_items(obj):
    if isinstance(obj, dict) or isinstance(obj, OrderedDict) or hasattr(obj, "items"):
        try:
            for k,v in obj.items(): yield k,v
            return
        except Exception: pass
    if isinstance(obj, (list, tuple, set, frozenset)):
        if isinstance(obj, (set, frozenset)):
            for i, v in enumerate(sorted(list(obj), key=lambda z: repr(z))): yield i, v
        else:
            for i, v in enumerate(obj): yield i, v
        return
    if torch is not None and isinstance(obj, (torch.nn.Module,)):
        try:
            for k,v in obj.state_dict().items(): yield k,v
            return
        except Exception: pass
    if hasattr(obj, "__dict__"):
        for k,v in obj.__dict__.items(): yield k,v
def _type_name(x):
    t = type(x); 
    return t.__name__ if t.__module__ == "builtins" else f"{t.__module__}.{t.__name__}"

def walk(obj: Any, name="root", *, max_depth=8, truncate=120, summarize=False, _seen=None, _depth=0, _prefix_parts=None):
    if _seen is None: _seen=set()
    if _prefix_parts is None: _prefix_parts=[]
    lines=[]
    obj_id=id(obj); cyc = obj_id in _seen
    head = f"{name} : {_type_name(obj)}"
    extra=_shape_dtype_device(obj); 
    if extra: head += f" [{extra}]"
    if cyc:
        lines.append(("".join(_prefix_parts)+head+"  (↩ seen)")); return lines
    _seen.add(obj_id); lines.append(("".join(_prefix_parts)+head))
    if _depth >= max_depth:
        lines.append(("".join(_prefix_parts)+"└── … (max depth reached)")); return lines
    leaf = (int,float,complex,bool,str,bytes)
    array_like = ((np.ndarray,) if np is not None else tuple()) + ((torch.Tensor,) if torch is not None else tuple()) + ((jnp.ndarray,) if jnp is not None else tuple())
    pandas_like = ( (pd.DataFrame, pd.Series) if pd is not None else tuple() )
    mapping_like=(dict, OrderedDict); seq_like=(list, tuple, set, frozenset)
    if isinstance(obj, leaf + array_like + pandas_like):
        if summarize:
            stats=_summ_stats(obj)
            if stats: lines.append(("".join(_prefix_parts)+f"└── {stats}"))
        else:
            if isinstance(obj, (str,bytes)):
                lines.append(("".join(_prefix_parts)+f"└── value={_describe_scalar(obj, truncate)}"))
        return lines
    if _is_namedtuple_instance(obj):
        items=list(zip(obj._fields, list(obj))); n=len(items)
        for i,(k,v) in enumerate(items):
            last=(i==n-1); branch="└── " if last else "├── "; cont="    " if last else "│   "
            lines.extend(walk(v, name=f"{k}", max_depth=max_depth, truncate=truncate, summarize=summarize, _seen=_seen, _depth=_depth+1, _prefix_parts=_prefix_parts+[branch]))
            _prefix_parts[-1:]=[cont]
        return lines
    if isinstance(obj, mapping_like) or hasattr(obj, "items"):
        try: items=list(_iter_items(obj))
        except Exception: items=[]
        n=len(items)
        for i,(k,v) in enumerate(items):
            last=(i==n-1); branch="└── " if last else "├── "; cont="    " if last else "│   "
            key_repr=_short(repr(k), truncate)
            lines.extend(walk(v, name=f"[{key_repr}]", max_depth=max_depth, truncate=truncate, summarize=summarize, _seen=_seen, _depth=_depth+1, _prefix_parts=_prefix_parts+[branch]))
            _prefix_parts[-1:]=[cont]
        return lines
    if isinstance(obj, seq_like):
        items=list(_iter_items(obj)); n=len(items)
        for i,(k,v) in enumerate(items):
            last=(i==n-1); branch="└── " if last else "├── "; cont="    " if last else "│   "
            lines.extend(walk(v, name=f"[{k}]", max_depth=max_depth, truncate=truncate, summarize=summarize, _seen=_seen, _depth=_depth+1, _prefix_parts=_prefix_parts+[branch]))
            _prefix_parts[-1:]=[cont]
        return lines
    if torch is not None and isinstance(obj, torch.nn.Module):
        try:
            return walk(obj.state_dict(), name=name+".state_dict()", max_depth=max_depth, truncate=truncate, summarize=summarize, _seen=_seen, _depth=_depth, _prefix_parts=_prefix_parts)
        except Exception: pass
    if hasattr(obj, "__dict__"):
        attrs=list(obj.__dict__.items()); n=len(attrs)
        for i,(k,v) in enumerate(attrs):
            last=(i==n-1); branch="└── " if last else "├── "; cont="    " if last else "│   "
            lines.extend(walk(v, name=f".{k}", max_depth=max_depth, truncate=truncate, summarize=summarize, _seen=_seen, _depth=_depth+1, _prefix_parts=_prefix_parts+[branch]))
            _prefix_parts[-1:]=[cont]
        return lines
    lines.append(("".join(_prefix_parts)+f"└── value={_short(repr(obj), truncate)}")); return lines

def load_any(path: str, encoding: str = "auto", force_torch: bool = False):
    if force_torch and torch is None:
        print("[warn] --use-torch-load set but torch is not installed; falling back to pickle.")
    if (force_torch or (torch is not None and os.path.splitext(path)[1] in (".pt",".pth",".ckpt"))):
        try:
            return torch.load(path, map_location="cpu" if torch is not None else None)
        except Exception as e:
            print(f"[info] torch.load failed: {e}. Will try pickle.")
    import pickle
    encs=[None,"latin1","utf-8"] if encoding=="auto" else ([None] if encoding=="none" else [encoding])
    last=None
    for enc in encs:
        try:
            with open(path,"rb") as f:
                return pickle.load(f) if enc is None else pickle.load(f, encoding=enc)
        except Exception as e:
            last=e; continue
    raise RuntimeError(f"Failed to load pickle: {last}")

def inspect_pickle(path, max_depth=8, truncate=120, summarize=False, encoding="auto", use_torch_load=False):
    obj = load_any(path, encoding=encoding, force_torch=use_torch_load)
    print(f"# Structure of: {path}\n")
    for ln in walk(obj, name="root", max_depth=max_depth, truncate=truncate, summarize=summarize):
        print(ln)
    return obj  # return the loaded object for further use


In [2]:
import os

def print_tree(startpath, max_depth=None):
    """
    Print a tree structure of files/folders starting from `startpath`.

    Parameters
    ----------
    startpath : str
        The root folder to print.
    max_depth : int or None
        Limit depth of recursion (None = unlimited).
    """
    startpath = os.path.abspath(startpath)
    for root, dirs, files in os.walk(startpath):
        depth = root[len(startpath):].count(os.sep)
        if max_depth is not None and depth > max_depth:
            continue
        indent = " " * 4 * depth
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * 4 * (depth + 1)
        for f in files:
            print(f"{subindent}{f}")

# Example usage:
print_tree("/path/to/your/folder", max_depth=3)


In [4]:
import os
from typing import Iterable, List, Union

def list_files_with_ext(
    root: str,
    exts: Union[str, Iterable[str]],
    *,
    case_insensitive: bool = True,
    follow_symlinks: bool = False,
    include_hidden: bool = True,
) -> List[str]:
    """
    Return absolute paths of all files under `root` whose extension matches `exts`.

    Parameters
    ----------
    root : str
        Directory to search (walks recursively).
    exts : str | Iterable[str]
        File extension(s), with or without leading dot. Examples: "pt", ".pt", ["pkl",".pt"].
    case_insensitive : bool
        If True, match extensions case-insensitively.
    follow_symlinks : bool
        If True, os.walk will follow directory symlinks.
    include_hidden : bool
        If False, skip hidden files and hidden directories (names starting with '.').
    """
    # normalize extensions
    if isinstance(exts, str):
        exts = [exts]
    norm = []
    for e in exts:
        e = e.strip()
        if not e:
            continue
        if not e.startswith("."):
            e = "." + e
        norm.append(e.lower() if case_insensitive else e)
    norm = tuple(norm)

    results: List[str] = []
    root_abs = os.path.abspath(root)

    for dirpath, dirnames, filenames in os.walk(root_abs, followlinks=follow_symlinks):
        if not include_hidden:
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            filenames = [f for f in filenames if not f.startswith(".")]

        for fname in filenames:
            ext = os.path.splitext(fname)[1]
            key = ext.lower() if case_insensitive else ext
            if key in norm:
                results.append(os.path.join(dirpath, fname))

    results.sort()
    return results

# Example:
# files = list_files_with_ext("/path/to/folder", [".pt", "pkl"])
# print("\n".join(files))


In [13]:
print_tree("pickle_files")

pickle_files/
    gradient_test_exponax_nu0.0005_nsamples1.pkl


In [14]:
files = list_files_with_ext("pickle_files", [".pt", "pkl"])
files

['/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/attack_result_viz/pickle_files/gradient_test_exponax_nu0.0005_nsamples1.pkl']

In [15]:
for file in files:   
    inspect_pickle(file)

# Structure of: /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/attack_result_viz/pickle_files/gradient_test_exponax_nu0.0005_nsamples1.pkl

root : dict
├── [('norm_2', 'index_0', 'numsteps_100', 'epsilon_20', 'alpha_1.00000')] : dict
├── ├── ['step_update'] : dict
├── ├── ├── ['step_0'] : dict
├── ├── ├── └── ['loss_metrics'] : dict
├── ├── ├── └── ├── ['with_solver'] : float
├── ├── ├── │   ├── ['without_solver'] : float
├── ├── ├── │   ├── ['approximate'] : float
├── ├── ├── │   └── ['approximate_surrogate'] : float
├── │   ├── ['step_1'] : dict
├── │   ├── ├── ['values'] : dict
├── │   ├── ├── ├── ['a_values'] : dict
├── │   ├── ├── ├── ├── ['a'] : numpy.ndarray [shape=(1024,), dtype=float64]
├── │   ├── ├── │   ├── ['a_with'] : numpy.ndarray [shape=(1024,), dtype=float64]
├── │   ├── ├── │   ├── ['a_without'] : numpy.ndarray [shape=(1024,), dtype=float64]
├── │   ├── ├── │   └── ['a_approximate'] : numpy.ndarray [shape=(1024,), dtype=float64]
├── │   ├── │  

In [16]:
import pickle

# --- Reading a pickle file ---
with open(file, "rb") as f:   # "rb" = read in binary mode
    data = pickle.load(f)

In [17]:
data.keys()

dict_keys([('norm_2', 'index_0', 'numsteps_100', 'epsilon_20', 'alpha_1.00000'), ('norm_2', 'index_0', 'numsteps_100', 'epsilon_50', 'alpha_1.00000'), ('norm_2', 'index_0', 'numsteps_100', 'epsilon_40', 'alpha_2.00000'), ('norm_2', 'index_0', 'numsteps_100', 'epsilon_100', 'alpha_2.00000')])

In [12]:
len(data)

480